Explore hyper-parameters

Training YOLOv8n with Real Waste images in Colab.
This will capture training time.

In [13]:
import os
print(os.getcwd())

/Users/juliareinhart/Documents/CSCI611AppliedMachineLearning/BranchJuliaFinalProject/waste-classification-ml


Because Colab uses temporarly environments, we need to install ultralytics every time I need to downlaod a model and train it.

In [14]:
#skipping this cell bc i am running locally on my mac so i can skip the content code

#import os
#print(os.listdir('/content'))

Mount google drive in this environment

In [15]:
# I am running this locally on my Mac so I can skip these lines

#from google.colab import drive
#drive.mount('/content/drive')

In [16]:
!pip install ultralytics

Check if we have GPU working.

In [17]:
import torch
print(torch.cuda.is_available())
# On Mac, CUDA usually returns False. That is okay because training can use MPS.
# Should return: True

False


In [18]:
#test to make sure input and output directories exist

from pathlib import Path

SOURCE_DIR = Path("/data/realwaste-main/RealWaste")

print("SOURCE_DIR exists:", SOURCE_DIR.exists())
print("Folders inside SOURCE_DIR:")

if SOURCE_DIR.exists():
    for item in SOURCE_DIR.iterdir():
        if item.is_dir():
            print(item.name)

SOURCE_DIR exists: False
Folders inside SOURCE_DIR:


Create stratified YOLO classification dataset

Training Block

In [19]:
#switching from using Yaml Classification to a stratified split
#no longer a random split but a stratified split

from pathlib import Path
import shutil
from sklearn.model_selection import train_test_split
from collections import Counter

SOURCE_DIR = Path("data/realwaste-main/RealWaste")
OUTPUT_DIR = Path("outputs_for_YOLO_classification_stratified_and_data_augmented_dataset")

train_ratio = 0.70
seed = 42

image_extensions = {".jpg", ".jpeg", ".png", ".webp"}

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

for split in ["train", "val", "test"]:
    (OUTPUT_DIR / split).mkdir(parents=True, exist_ok=True)

image_paths = []
targets = []

class_names = sorted([p.name for p in SOURCE_DIR.iterdir() if p.is_dir()])

for class_name in class_names:
    class_folder = SOURCE_DIR / class_name
    for img_path in class_folder.iterdir():
        if img_path.suffix.lower() in image_extensions:
            image_paths.append(img_path)
            targets.append(class_name)

print("Classes found:", class_names)
print("Total images:", len(image_paths))
print("Original class counts:", Counter(targets))

train_paths, temp_paths, train_targets, temp_targets = train_test_split(
    image_paths,
    targets,
    train_size=train_ratio,
    stratify=targets,
    random_state=seed
)

val_paths, test_paths, val_targets, test_targets = train_test_split(
    temp_paths,
    temp_targets,
    test_size=0.5,
    stratify=temp_targets,
    random_state=seed
)

def copy_split(paths, labels, split_name):
    for src_path, class_name in zip(paths, labels):
        dest_dir = OUTPUT_DIR / split_name / class_name
        dest_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src_path, dest_dir / src_path.name)

copy_split(train_paths, train_targets, "train")
copy_split(val_paths, val_targets, "val")
copy_split(test_paths, test_targets, "test")

print("Done creating stratified YOLO classification folders!")
print("Train:", len(train_paths), Counter(train_targets))
print("Val:", len(val_paths), Counter(val_targets))
print("Test:", len(test_paths), Counter(test_targets))

Classes found: ['Cardboard', 'Food Organics', 'Glass', 'Metal', 'Miscellaneous Trash', 'Paper', 'Plastic', 'Textile Trash', 'Vegetation']
Total images: 4752
Original class counts: Counter({'Plastic': 921, 'Metal': 790, 'Paper': 500, 'Miscellaneous Trash': 495, 'Cardboard': 461, 'Vegetation': 436, 'Glass': 420, 'Food Organics': 411, 'Textile Trash': 318})
Done creating stratified YOLO classification folders!
Train: 3326 Counter({'Plastic': 645, 'Metal': 553, 'Paper': 350, 'Miscellaneous Trash': 346, 'Cardboard': 323, 'Vegetation': 305, 'Glass': 294, 'Food Organics': 288, 'Textile Trash': 222})
Val: 713 Counter({'Plastic': 138, 'Metal': 119, 'Paper': 75, 'Miscellaneous Trash': 74, 'Cardboard': 69, 'Vegetation': 66, 'Glass': 63, 'Food Organics': 61, 'Textile Trash': 48})
Test: 713 Counter({'Plastic': 138, 'Metal': 118, 'Paper': 75, 'Miscellaneous Trash': 75, 'Cardboard': 69, 'Vegetation': 65, 'Glass': 63, 'Food Organics': 62, 'Textile Trash': 48})


In [20]:
#verification after cell 13 to ensure INPUT and OUTPUT directories exist

from pathlib import Path
from collections import Counter

OUTPUT_DIR = Path("outputs_for_YOLO_classification_stratified_and_data_augmented_dataset")

for split in ["train", "val", "test"]:
    split_dir = OUTPUT_DIR / split
    print(f"\n{split.upper()} exists:", split_dir.exists())

    class_counts = {}
    for class_dir in sorted(split_dir.iterdir()):
        if class_dir.is_dir():
            count = len(list(class_dir.glob("*")))
            class_counts[class_dir.name] = count

    print(class_counts)


TRAIN exists: True
{'Cardboard': 323, 'Food Organics': 288, 'Glass': 294, 'Metal': 553, 'Miscellaneous Trash': 346, 'Paper': 350, 'Plastic': 645, 'Textile Trash': 222, 'Vegetation': 305}

VAL exists: True
{'Cardboard': 69, 'Food Organics': 61, 'Glass': 63, 'Metal': 119, 'Miscellaneous Trash': 74, 'Paper': 75, 'Plastic': 138, 'Textile Trash': 48, 'Vegetation': 66}

TEST exists: True
{'Cardboard': 69, 'Food Organics': 62, 'Glass': 63, 'Metal': 118, 'Miscellaneous Trash': 75, 'Paper': 75, 'Plastic': 138, 'Textile Trash': 48, 'Vegetation': 65}


In [21]:

from ultralytics import YOLO
import torch
from datetime import datetime

# Start time
start_time = datetime.now()
print("Training started at:", start_time.strftime("%Y-%m-%d %H:%M:%S"))
# ======> Train the model
def main():

    # 1. Check for NVIDIA GPU (Colab/PC)
    # 2. Check for Apple Silicon (Mac)
    # 3. Default to CPU
    device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

    print(f"I am training on: {device.upper()}")

    ### Smaller model.
    model = YOLO("yolov8n-cls.pt")

    model.train(
        ### data="realwaste-yolo/RealWaste/data.yaml",
        ### '/content/RealWaste/data.yaml'
        data="outputs_for_YOLO_classification_stratified_and_data_augmented_dataset",
        epochs=10, ###==> Hyper-parameters
        lr0=0.005,
        lrf=0.1,
        momentum=0.9,
        weight_decay=0.0001,
        ###warmup_epochs=5,
        ###warmup_momentum=0.8,
        ###warmup_bias_lr=0.1, ###<==
        imgsz=224,
        batch=16,
        patience=25,
        device=device,
        workers=0,  # safer in Jupyter if needed
        project="runs",
        name="yolov8n_cls_stratified_aug_10epochs",
        degrees=8,
        scale=0.15,
        fliplr=0.5,
        hsv_h=0.01,
        hsv_s=0.05,
        hsv_v=0.05,
        perspective=0.0,
    )

if __name__ == "__main__":
    main()
# <=====

# End time
end_time = datetime.now()
print("Training ended at:", end_time.strftime("%Y-%m-%d %H:%M:%S"))

# Elapsed time
elapsed = end_time - start_time

# Format nicely (hours, minutes, seconds)
hours, remainder = divmod(elapsed.total_seconds(), 3600)
minutes, seconds = divmod(remainder, 60)

print(f"Elapsed time: {int(hours)}h {int(minutes)}m {int(seconds)}s")

Training started at: 2026-05-06 21:33:11
I am training on: MPS
New https://pypi.org/project/ultralytics/8.4.47 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.46 🚀 Python-3.11.15 torch-2.11.0 MPS (Apple M4 Max)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=outputs_for_YOLO_classification_stratified_and_data_augmented_dataset, degrees=8, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.01, hsv_s=0.05, hsv_v=0.05, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.005, lrf=0.1, mask_ratio=4, max_det=300, mixup=

In [22]:
#code to actually test the model, and see what the test accuracy is
#very important was missing before but we need test accuracy after each run to verify whether or not the model is improving or not

from ultralytics import YOLO

# Load best trained model
model = YOLO("runs/classify/runs/yolov8n_cls_stratified_aug_10epochs-3/weights/best.pt")

# Evaluate on test dataset
metrics = model.val(
    data="outputs_for_YOLO_classification_stratified_and_data_augmented_dataset",
    split="test"
)

print("\nEvaluation Metrics:")
print(metrics)

print(f"\nTop-1 Accuracy: {metrics.top1:.4f}")
print(f"Top-5 Accuracy: {metrics.top5:.4f}")

Ultralytics 8.4.46 🚀 Python-3.11.15 torch-2.11.0 CPU (Apple M4 Max)
YOLOv8n-cls summary (fused): 30 layers, 1,446,409 parameters, 0 gradients, 3.3 GFLOPs
train: /Users/juliareinhart/Documents/CSCI611AppliedMachineLearning/BranchJuliaFinalProject/waste-classification-ml/outputs_for_YOLO_classification_stratified_and_data_augmented_dataset/train... found 3326 images in 9 classes ✅ 
val: /Users/juliareinhart/Documents/CSCI611AppliedMachineLearning/BranchJuliaFinalProject/waste-classification-ml/outputs_for_YOLO_classification_stratified_and_data_augmented_dataset/val... found 713 images in 9 classes ✅ 
test: /Users/juliareinhart/Documents/CSCI611AppliedMachineLearning/BranchJuliaFinalProject/waste-classification-ml/outputs_for_YOLO_classification_stratified_and_data_augmented_dataset/test... found 713 images in 9 classes ✅ 
test: Fast image access ✅ (ping: 0.0±0.0 ms, read: 676.3±70.3 MB/s, size: 190.6 KB)
test: Scanning /Users/juliareinhart/Documents/CSCI611AppliedMachineLearning/BranchJ